# Session · State · Memory

멀티턴 대화는 **세션**의 **Event** 목록과 **state**로 이어집니다.

- 소개: [Session, State, and Memory](https://google.github.io/adk-docs/sessions/)
- 세션 객체: [The Session object](https://google.github.io/adk-docs/sessions/session/)

코드 예제는 **Gemini** + Google AI Studio **무료 API 키**를 사용합니다.

## 핵심 개념

| 개념 | 설명 |
|------|------|
| **Session** | 한 대화 스레드 (`events`, `state`) |
| **State** | 이 대화 안의 임시 데이터 |
| **Memory** | 세션을 넘는 장기 저장 (`MemoryService`) |

## SessionService

- **InMemorySessionService** — 개발용  
- **DatabaseSessionService** — SQLite 등 영속화  
- **VertexAiSessionService** — Vertex 연동 시

## 1) InMemorySessionService — 세션 필드 확인

`create_session`으로 만든 객체의 `id`, `app_name`, `user_id`, `state`, `events`를 봅니다.

In [ ]:
from google.adk.sessions import InMemorySessionService

# 세션 저장·조회·삭제를 담당하는 서비스 객체
svc = InMemorySessionService()

# app_name + user_id 로 "누구의 어떤 앱 대화"인지 구분
# state: 대화 중 유지할 초기 키-값 (장바구니, 단계 등)
sess = await svc.create_session(
    app_name="my_app",
    user_id="example_user",
    state={"cart": []},
)

print("id:", sess.id)
print("app_name:", sess.app_name)
print("user_id:", sess.user_id)
print("state:", dict(sess.state))
print("events (초기):", len(sess.events))  # 대화 전이면 비어 있음

await svc.delete_session(
    app_name=sess.app_name, user_id=sess.user_id, session_id=sess.id
)

## 2) Runner — 같은 `session_id`로 멀티턴

같은 사용자·세션으로 `run_async`를 반복 호출하면 이전 턴 맥락이 이어질 수 있습니다.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# 멀티턴 예제에서 Gemini 호출을 위해 키 로드
load_dotenv(Path(".env").resolve())
load_dotenv(Path("..") / ".env")
assert os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY"), (
    ".env 에 GOOGLE_API_KEY 또는 GEMINI_API_KEY 를 설정하세요."
)

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP = "session_memory_lesson"
USER = "learner"
SESSION = "thread_1"

agent = LlmAgent(
    model="gemini-2.0-flash",
    name="memory_aware",
    instruction=(
        "한국어로 답하세요. 사용자가 말한 이름을 기억하고 다음 턴에서 인사할 때 사용하세요."
    ),
)

session_service = InMemorySessionService()
await session_service.create_session(
    app_name=APP, user_id=USER, session_id=SESSION, state={}
)
runner = Runner(agent=agent, app_name=APP, session_service=session_service)


async def say(text: str) -> str:
    """한 턴의 사용자 입력을 보내고, 모델 최종 텍스트만 문자열로 돌려줍니다."""
    msg = types.Content(role="user", parts=[types.Part(text=text)])
    out = ""
    async for event in runner.run_async(
        user_id=USER, session_id=SESSION, new_message=msg
    ):
        if event.is_final_response() and event.content and event.content.parts:
            t = event.content.parts[0].text
            if t:
                out = t
    return out


print("턴1:", await say("내 이름은 민수야."))
print("턴2:", await say("내 이름이 뭐라고 했지?"))

# get_session: 디스크가 아닌 서비스 메모리에서 현재 세션 스냅샷 읽기
loaded = await session_service.get_session(
    app_name=APP, user_id=USER, session_id=SESSION
)
print("저장된 이벤트 수:", len(loaded.events))

await session_service.delete_session(
    app_name=APP, user_id=USER, session_id=SESSION
)

## 3) DatabaseSessionService — SQLite 영속화

비동기 드라이버가 필요합니다. 예: `sqlite+aiosqlite:///...`

- [스키마 마이그레이션](https://google.github.io/adk-docs/sessions/session/migrate/)

In [ ]:
import os
from pathlib import Path

from google.adk.sessions import DatabaseSessionService

# SQLite 파일 경로 (resolve()로 절대 경로 권장)
db_path = Path("./_adk_session_demo.db").resolve()
# aiosqlite: 비동기 SQLite 드라이버 (일반 sqlite:// 와 다름)
db_url = f"sqlite+aiosqlite:///{db_path.as_posix()}"

db_service = DatabaseSessionService(db_url=db_url)

s = await db_service.create_session(
    app_name="db_app",
    user_id="u_db",
    state={"note": "persisted"},
)
print("DB 세션 id:", s.id, "state:", dict(s.state))

await db_service.delete_session(
    app_name="db_app", user_id="u_db", session_id=s.id
)

# 연습용: 파일을 남기지 않으려면 삭제
if db_path.exists():
    os.remove(db_path)
    print("데모 DB 파일 삭제됨")

## (참고) MemoryService

장기 메모리는 별도 `MemoryService`로 검색·저장합니다. 자세한 구성은 [공식 문서](https://google.github.io/adk-docs/sessions/)를 참고하세요.